In [10]:
from utils.langchain_model import get_singleton_client

model=get_singleton_client(llm_provider="bailing")  # bailing  longcat

model.invoke('介绍一下你自己')

AIMessage(content='你好！我是由蚂蚁集团研发的语言模型，名为“百灵”。我是一个基于深度学习和自然语言处理技术的语言模型，可以回答各种问题，提供帮助和支持。我具有广泛的知识和信息，可以处理多种语言和任务，例如文本生成、问答、翻译、摘要等。我致力于为用户提供准确、有用和友好的服务。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 62, 'prompt_tokens': 20, 'total_tokens': 82, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'Ling-2.6-1T', 'system_fingerprint': None, 'id': '0a6444e617827227363163205793', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f128e-2841-7fa0-9c87-783c7321a5bc-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 20, 'output_tokens': 62, 'total_tokens': 82, 'input_token_details': {}, 'output_token_details': {}})

In [11]:
from dotenv import load_dotenv
from langchain_ollama.embeddings import OllamaEmbeddings
import os

load_dotenv()

OLLAMA_BASEURL = os.getenv("OLLAMA_BASEURL")
OLLAMA_MODEL_NAME = os.getenv("OLLAMA_MODEL_NAME")
OLLAMA_EMBEDDING_MODEL_NAME = (
    "bge-m3:latest"  # os.getenv("OLLAMA_EMBEDDING_MODEL_NAME")
)
tavily_api_key = os.getenv("TAVILY_API_KEY")

embeddings = OllamaEmbeddings(
    model=OLLAMA_EMBEDDING_MODEL_NAME, base_url=OLLAMA_BASEURL
)

In [12]:
from typing import TypedDict, List, Literal  
from langchain_core.documents import Document  
from langgraph.graph import StateGraph, END  
from langchain_community.vectorstores import FAISS  
from langchain_community.tools.tavily_search import TavilySearchResults  
from langchain_community.graphs import Neo4jGraph  
from pydantic import BaseModel  

llm = model

# ── Agent 状态——在所有节点间持久保存 ────────────  
class AgentState(TypedDict):  
    question:       str  
    route:          str                   # vector | graph | web | direct  
    documents:      List[Document]  
    generation:     str  
    rewrite_count:  int  
    grade_results:  List[str]  

# ── 工具初始化 ──────────────────────────────────────────────  
vector_store = FAISS.load_local("faiss_index", 
                                embeddings,  
                                allow_dangerous_deserialization=True,  # 反序列化 pickle 需要显式确认
                                )  
vector_retriever = vector_store.as_retriever(search_kwargs={"k": 4})  

neo4j_graph = Neo4jGraph(url="bolt://localhost:7687",  
                          username="neo4j", password="your_secure_password")  
web_search = TavilySearchResults(max_results=3, tavily_api_key=tavily_api_key)

In [ ]:
class RouteDecision(BaseModel):  
    route: Literal["vector", "graph", "web", "direct"]  
    reasoning: str  


ROUTER_PROMPT = """You are a query router for a hybrid RAG system.  
Classify the query into ONE category:  

- vector: factual questions answerable from internal documents  
  (policies, reports, product info, static knowledge)  
- graph: questions about relationships between entities  
  (how X connects to Y, who knows whom, supplier chains)    
- web: requires real-time or current information  
  (news, stock prices, today's events, recent releases)  
- direct: simple calculations or general knowledge the LLM already knows  

Query: {question}"""  

def router_node(state: AgentState) -> AgentState:  
    router_llm = llm.with_structured_output(RouteDecision)  

    decision = router_llm.invoke(  
        ROUTER_PROMPT.format(question=state["question"])  
    )  
    print(f"→ Router: {decision.route} | {decision.reasoning}")  
    return {**state, "route": decision.route}  

# 条件边——路由到对应的检索节点  
def route_edge(state: AgentState) -> str:  
     return state["route"]   # "vector" | "graph" | "web" | "direct"

In [14]:
# ── Vector Retriever 节点 ────────────────────────────────
def vector_node(state: AgentState) -> AgentState:
    """从FAISS向量库检索相关文档"""
    docs = vector_retriever.invoke(state["question"])
    print(f"→ Vector检索到 {len(docs)} 个文档")
    return {**state, "documents": docs}

# ── Graph Retriever 节点 ────────────────────────────────
def graph_node(state: AgentState) -> AgentState:
    """从Neo4j图数据库检索关系信息"""
    # 使用LLM将自然语言转为Cypher查询
    cypher_query = llm.invoke(
        f"Convert this question to a Cypher query: {state['question']}"
    ).content
    
    # 执行查询
    results = neo4j_graph.query(cypher_query)
    
    # 将结果转为Document对象
    docs = [Document(page_content=str(result)) for result in results]
    print(f"→ Graph检索到 {len(docs)} 个文档")
    return {**state, "documents": docs}

# ── Web Retriever 节点 ──────────────────────────────────
def web_node(state: AgentState) -> AgentState:
    """从Tavily进行网络搜索"""
    results = web_search.invoke(state["question"])
    docs = [
        Document(
            page_content=f"{r['content']}\nSource: {r.get('url', '')}",
            metadata={"url": r.get("url", "")}
        )
        for r in results
    ]
    print(f"→ Web检索到 {len(docs)} 个文档")
    return {**state, "documents": docs}

# ── Direct 节点（不需要检索）─────────────────────────────
def direct_node(state: AgentState) -> AgentState:
    """直接调用LLM回答，不需要检索"""
    answer = llm.invoke(state["question"]).content
    return {**state, "generation": answer, "documents": []}

In [15]:
# ── Grader ───────────────────────────────────────────────  
class GradeDoc(BaseModel):  
    score: Literal["relevant", "irrelevant"]  

grader_llm = llm.with_structured_output(GradeDoc)  

def grader_node(state: AgentState) -> AgentState:  
    grades = []  
    for doc in state["documents"]:  
        result = grader_llm.invoke(  
            f"Question: {state['question']}\nDocument: {doc.page_content[:400]}\n"  
            "Is this document relevant to answering the question? Score: relevant/irrelevant"  
        )  
        grades.append(result.score)  
    return {**state, "grade_results": grades}  

def grade_edge(state: AgentState) -> str:  
    relevant = sum(1 for g in state["grade_results"] if g == "relevant")  
    if relevant > 0:  
        return "generate"  
    elif state["rewrite_count"] < 3:  
        return "rewrite"  
    else:  
        return "web_fallback"   # 三次失败后的最终兜底  

# ── Rewriter ─────────────────────────────────────────────  
def rewriter_node(state: AgentState) -> AgentState:  
    rewritten = llm.invoke(  
        f"The query '{state['question']}' returned no relevant results. "  
        "Rewrite it to be more specific and searchable. Return only the rewritten query."  
    ).content  
    print(f"→ Rewriter: '{state['question']}' → '{rewritten}'")  
    return {**state, "question": rewritten,  
            "rewrite_count": state["rewrite_count"] + 1}  

# ── Generator ────────────────────────────────────────────  
def generator_node(state: AgentState) -> AgentState:  
    context = "\n\n".join(d.page_content for d in state["documents"]  
                          if "relevant" in state.get("grade_results",[]))  
    answer = llm.invoke(  
        f"Answer using only the context below.\n\nContext:\n{context}\n\nQuestion: {state['question']}"  
    ).content  
    return {**state, "generation": answer}  

# ── Hallucination Checker ────────────────────────────────  
class HallucinationCheck(BaseModel):  
    grounded: Literal["yes", "no"]  

halluc_llm = llm.with_structured_output(HallucinationCheck)  

def hallucination_node(state: AgentState) -> AgentState:  
    context = "\n\n".join(d.page_content for d in state["documents"])  
    result = halluc_llm.invoke(  
        f"Context:\n{context}\n\nAnswer:\n{state['generation']}\n\n"  
        "Is the answer fully supported by the context? grounded: yes/no"  
    )  
    return {**state, "hallucination_check": result.grounded}  

def halluc_edge(state: AgentState) -> str:  
     return "end" if state.get("hallucination_check") == "yes" else "regenerate"

In [16]:
# ── 构建 LangGraph 状态机 ────────────────────────────────  
workflow = StateGraph(AgentState)  

# 添加节点  
workflow.add_node("router",       router_node)  
workflow.add_node("vector",       vector_node)  
workflow.add_node("graph",        graph_node)  
workflow.add_node("web",          web_node)  
workflow.add_node("direct",       direct_node)  
workflow.add_node("grader",       grader_node)  
workflow.add_node("rewriter",     rewriter_node)  
workflow.add_node("generator",    generator_node)  
workflow.add_node("hallucination",hallucination_node)  

# 入口节点  
workflow.set_entry_point("router")  

# router → 检索器（条件边）  
workflow.add_conditional_edges("router", route_edge, {  
    "vector": "vector",  
    "graph":  "graph",  
    "web":    "web",  
    "direct": "direct",  
})  

# 检索器 → grader  
for node in ["vector", "graph", "web"]:  
    workflow.add_edge(node, "grader")  

# grader → 生成 / 改写 / web 兜底  
workflow.add_conditional_edges("grader", grade_edge, {  
    "generate":     "generator",  
    "rewrite":      "rewriter",  
    "web_fallback": "web",  
})  

# rewriter → 回到 router  
workflow.add_edge("rewriter", "router")  

# generator → hallucination 检测  
workflow.add_edge("generator", "hallucination")  

# hallucination 检测 → 结束或重新生成  
workflow.add_conditional_edges("hallucination", halluc_edge, {  
    "end":        END,  
    "regenerate": "generator",  
})  
workflow.add_edge("direct", END)  

# 编译并运行  
agent = workflow.compile()  

# agent

In [ ]:
result = agent.invoke({  
    "question":      "什么是向量检索?",  
    "rewrite_count": 0,  
    "documents":     [],  
    "grade_results": [],  
    "generation":    "",  
    "route":         "",  
})  
print(result["generation"])